# Advanced Machine Learning Training Pipeline Demo

This notebook demonstrates how to use the advanced machine learning training pipeline for creating and training transformer-based models. The pipeline includes components for data loading, preprocessing, augmentation, model architecture, training, evaluation, hyperparameter tuning, and model optimization.

## Setup

First, let's clone the repository and install the required dependencies.

In [ ]:
# Clone the repository
!git clone https://github.com/backup-bdg-6/datasets.git ml-training-pipeline
%cd ml-training-pipeline

In [ ]:
# Install required dependencies
!pip install torch transformers datasets accelerate bitsandbytes nltk pandas numpy pyyaml wandb matplotlib seaborn scikit-learn rouge-score

In [ ]:
# Optional: Install Ray Tune for hyperparameter optimization
!pip install "ray[tune]"

In [ ]:
# Optional: Install DeepSpeed for distributed training
!pip install deepspeed

## Create Required Directories

Let's make sure all the required directories exist.

In [ ]:
!mkdir -p src/data
!mkdir -p src/model
!mkdir -p src/utils
!mkdir -p output

## Create a Simple Configuration

Let's create a simplified configuration for our demo.

In [ ]:
%%writefile config/simple_config.yaml
# Simple configuration for Colab/Kaggle demo

# Project information
project_name: "transformer_model_demo"
output_dir: "./output"
seed: 42

# Model configuration
model:
  type: "decoder_only_transformer"
  size: "small"  # Using small model for Colab/Kaggle
  sizes:
    small:
      n_layers: 4
      n_heads: 8
      d_model: 256
      d_ff: 1024
      max_seq_length: 512
  dropout: 0.1
  attention:
    causal: true
    rotary_embedding: true

# Tokenizer configuration
tokenizer:
  type: "huggingface"
  name: "gpt2"
  vocab_size: 50257
  max_length: 512
  padding_side: "right"
  truncation_side: "right"
  add_bos_token: true
  add_eos_token: true

# Training configuration
training:
  active_stage: "finetune"  # Using a smaller dataset for demo
  stages:
    - name: "finetune"
      epochs: 2
      datasets:
        - name: "imdb"
          split: "train"
          streaming: false
          max_samples: 1000  # Limiting samples for Colab/Kaggle
        - name: "imdb"
          split: "test"
          streaming: false
          max_samples: 200  # Limiting samples for Colab/Kaggle
      learning_rate:
        initial: 2.0e-5
        min: 1.0e-6
        schedule: "linear"
        warmup_steps: 100
  
  # Optimizer settings
  optimizer:
    name: "adamw"
    weight_decay: 0.01
    beta1: 0.9
    beta2: 0.999
    eps: 1.0e-8
    use_8bit: false
  
  # Mixed precision settings
  mixed_precision: "fp16"
  gradient_checkpointing: false
  gradient_clipping: 1.0
  
  # Checkpointing settings
  checkpointing:
    save_steps: 100
    keep_last_n: 1
    save_optimizer_state: true
  
  # Evaluation settings
  evaluation:
    eval_steps: 100
    early_stopping:
      enabled: true
      patience: 3
      metric: "eval_loss"
      mode: "min"

# Data processing configuration
data_processing:
  preprocessing:
    normalize_unicode: true
    normalize_whitespace: true
    remove_html: true
    min_length: 10
    max_length: 512
  
  augmentation:
    enabled: true
    techniques:
      - name: "synonym_replacement"
        probability: 0.1
      - name: "random_deletion"
        probability: 0.05
  
  batching:
    train_batch_size: 8  # Smaller batch size for Colab/Kaggle
    eval_batch_size: 8
    gradient_accumulation_steps: 2
    dynamic_batching: false

# Evaluation configuration
evaluation:
  metrics:
    - "loss"
    - "perplexity"
    - "accuracy"
  
  generation:
    max_length: 64
    num_beams: 4
    temperature: 1.0
    top_p: 0.9
    top_k: 50
    do_sample: true

# Logging configuration
logging:
  use_wandb: false
  wandb_project: "transformer_model_demo"
  log_steps: 10
  save_logs: true

## Create Dummy Files for Missing Dependencies

Let's create dummy files for any missing dependencies to ensure the pipeline works correctly.

In [ ]:
%%writefile src/data/loaders.py
"""
Dataset loading utilities for the AI model training workflow.
"""

import os
import logging
import yaml
from typing import Dict, List, Optional, Union, Any

# Try to import optional dependencies
try:
    from datasets import load_dataset, Dataset, DatasetDict
    DATASETS_AVAILABLE = True
except ImportError:
    DATASETS_AVAILABLE = False

# Configure logging
logger = logging.getLogger(__name__)


class DatasetLoader:
    """
    A class to handle dataset loading from various sources.
    """
    
    def __init__(self, config_path: str):
        """
        Initialize the dataset loader.
        
        Args:
            config_path: Path to the configuration file
        """
        self.config_path = config_path
        self.config = self._load_config()
    
    def _load_config(self) -> Dict:
        """Load configuration from YAML file."""
        try:
            with open(self.config_path, 'r') as f:
                config = yaml.safe_load(f)
            return config
        except Exception as e:
            logger.error(f"Error loading configuration: {e}")
            return {}
    
    def load_dataset(
        self,
        dataset_name: str,
        subset: Optional[str] = None,
        split: Optional[str] = None,
        streaming: bool = False,
        max_samples: Optional[int] = None,
    ) -> Any:
        """
        Load a dataset from HuggingFace datasets, local files, or other sources.
        
        Args:
            dataset_name: Name of the dataset
            subset: Subset of the dataset
            split: Split of the dataset (train, validation, test)
            streaming: Whether to stream the dataset
            max_samples: Maximum number of samples to load
            
        Returns:
            Loaded dataset
        """
        if not DATASETS_AVAILABLE:
            logger.error("HuggingFace datasets library is not available. Please install with: pip install datasets")
            return None
        
        try:
            # Load dataset from HuggingFace
            if subset:
                dataset = load_dataset(dataset_name, subset, split=split, streaming=streaming)
            else:
                dataset = load_dataset(dataset_name, split=split, streaming=streaming)
            
            # Limit number of samples if specified
            if max_samples is not None and not streaming:
                dataset = dataset.select(range(min(max_samples, len(dataset))))
            
            return dataset
        except Exception as e:
            logger.error(f"Error loading dataset {dataset_name}: {e}")
            return None

In [ ]:
%%writefile src/data/preprocessors.py
"""
Data preprocessing utilities for the AI model training workflow.
"""

import re
import unicodedata
import logging
from typing import Dict, List, Optional, Union, Any, Callable

# Configure logging
logger = logging.getLogger(__name__)


class DataPreprocessor:
    """
    A class to handle data preprocessing.
    """
    
    def __init__(self, config: Dict):
        """
        Initialize the data preprocessor.
        
        Args:
            config: Configuration dictionary
        """
        self.config = config
        self.preprocessing_config = config.get('data_processing', {}).get('preprocessing', {})
    
    def process_text(self, text: str) -> str:
        """
        Process a text string.
        
        Args:
            text: Input text
            
        Returns:
            Processed text
        """
        if not text:
            return ""
        
        # Normalize Unicode if enabled
        if self.preprocessing_config.get('normalize_unicode', False):
            text = unicodedata.normalize('NFKC', text)
        
        # Remove HTML if enabled
        if self.preprocessing_config.get('remove_html', False):
            text = re.sub(r'<.*?>', ' ', text)
        
        # Normalize whitespace if enabled
        if self.preprocessing_config.get('normalize_whitespace', False):
            text = re.sub(r'\s+', ' ', text).strip()
        
        return text
    
    def process_dataset(self, dataset: Any) -> Any:
        """
        Process a dataset.
        
        Args:
            dataset: Input dataset
            
        Returns:
            Processed dataset
        """
        if dataset is None:
            return None
        
        # Process text column
        if hasattr(dataset, 'map'):
            # Determine text column
            text_column = None
            if hasattr(dataset, 'column_names'):
                if 'text' in dataset.column_names:
                    text_column = 'text'
                elif 'content' in dataset.column_names:
                    text_column = 'content'
                elif 'sentence' in dataset.column_names:
                    text_column = 'sentence'
            
            if text_column:
                # Process text column
                def process_example(example):
                    example[text_column] = self.process_text(example[text_column])
                    return example
                
                dataset = dataset.map(process_example)
                
                # Filter by length if specified
                min_length = self.preprocessing_config.get('min_length', 0)
                max_length = self.preprocessing_config.get('max_length', float('inf'))
                
                if min_length > 0 or max_length < float('inf'):
                    def length_filter(example):
                        return len(example[text_column]) >= min_length and len(example[text_column]) <= max_length
                    
                    dataset = dataset.filter(length_filter)
        
        return dataset

In [ ]:
%%writefile src/model/architecture.py
"""
Model architecture definitions for the AI model training workflow.
"""

import logging
from typing import Dict, Optional
import torch
import torch.nn as nn

# Try to import optional dependencies
try:
    from transformers import AutoModelForCausalLM, AutoConfig
    TRANSFORMERS_AVAILABLE = True
except ImportError:
    TRANSFORMERS_AVAILABLE = False

# Configure logging
logger = logging.getLogger(__name__)


def create_model_from_config(config: Dict) -> nn.Module:
    """
    Create a model from configuration.
    
    Args:
        config: Configuration dictionary
        
    Returns:
        Model instance
    """
    if not TRANSFORMERS_AVAILABLE:
        logger.error("Transformers library is not available. Please install with: pip install transformers")
        return None
    
    model_config = config.get('model', {})
    model_type = model_config.get('type', 'decoder_only_transformer')
    model_size = model_config.get('size', 'small')
    
    # Get model size parameters
    size_config = model_config.get('sizes', {}).get(model_size, {})
    n_layers = size_config.get('n_layers', 6)
    n_heads = size_config.get('n_heads', 8)
    d_model = size_config.get('d_model', 512)
    d_ff = size_config.get('d_ff', 2048)
    max_seq_length = size_config.get('max_seq_length', 1024)
    
    # Get other model parameters
    dropout = model_config.get('dropout', 0.1)
    
    # Create model based on type
    if model_type == 'decoder_only_transformer':
        # Create GPT-style model
        model_config = AutoConfig.from_pretrained(
            'gpt2',
            n_layer=n_layers,
            n_head=n_heads,
            n_embd=d_model,
            n_inner=d_ff,
            max_position_embeddings=max_seq_length,
            resid_pdrop=dropout,
            embd_pdrop=dropout,
            attn_pdrop=dropout,
        )
        
        model = AutoModelForCausalLM.from_config(model_config)
        logger.info(f"Created decoder-only transformer model with {n_layers} layers, {n_heads} heads, {d_model} dimensions")
    else:
        logger.error(f"Unsupported model type: {model_type}")
        return None
    
    return model

In [ ]:
%%writefile src/model/training.py
"""
Training utilities for the AI model training workflow.
"""

import os
import logging
import time
import math
from typing import Dict, List, Optional, Union, Any, Tuple, Callable
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, Dataset
from tqdm import tqdm

# Configure logging
logger = logging.getLogger(__name__)


class TrainingArguments:
    """
    Arguments for training.
    """
    
    def __init__(self, config: Dict, stage_name: str):
        """
        Initialize training arguments.
        
        Args:
            config: Configuration dictionary
            stage_name: Name of the training stage
        """
        self.config = config
        self.stage_name = stage_name
        
        # Get stage configuration
        stage_config = None
        for stage in config.get('training', {}).get('stages', []):
            if stage.get('name') == stage_name:
                stage_config = stage
                break
        
        if stage_config is None:
            logger.warning(f"Training stage {stage_name} not found in configuration")
            stage_config = {}
        
        # Set training arguments
        self.output_dir = config.get('output_dir', './output')
        self.epochs = stage_config.get('epochs', 3)
        
        # Learning rate settings
        lr_config = stage_config.get('learning_rate', {})
        self.learning_rate = lr_config.get('initial', 5e-5)
        self.min_learning_rate = lr_config.get('min', 0.0)
        self.lr_schedule = lr_config.get('schedule', 'linear')
        self.warmup_steps = lr_config.get('warmup_steps', 0)
        
        # Optimizer settings
        optimizer_config = config.get('training', {}).get('optimizer', {})
        self.optimizer_name = optimizer_config.get('name', 'adamw')
        self.weight_decay = optimizer_config.get('weight_decay', 0.01)
        self.beta1 = optimizer_config.get('beta1', 0.9)
        self.beta2 = optimizer_config.get('beta2', 0.999)
        self.eps = optimizer_config.get('eps', 1e-8)
        
        # Mixed precision settings
        self.mixed_precision = config.get('training', {}).get('mixed_precision', 'no')
        self.gradient_checkpointing = config.get('training', {}).get('gradient_checkpointing', False)
        self.gradient_clipping = config.get('training', {}).get('gradient_clipping', 1.0)
        
        # Batch size settings
        batching_config = config.get('data_processing', {}).get('batching', {})
        self.train_batch_size = batching_config.get('train_batch_size', 16)
        self.eval_batch_size = batching_config.get('eval_batch_size', 16)
        self.gradient_accumulation_steps = batching_config.get('gradient_accumulation_steps', 1)
        
        # Checkpointing settings
        checkpoint_config = config.get('training', {}).get('checkpointing', {})
        self.save_steps = checkpoint_config.get('save_steps', 0)
        self.keep_last_n = checkpoint_config.get('keep_last_n', 3)
        self.save_optimizer_state = checkpoint_config.get('save_optimizer_state', True)
        
        # Evaluation settings
        eval_config = config.get('training', {}).get('evaluation', {})
        self.eval_steps = eval_config.get('eval_steps', 0)
        
        # Early stopping settings
        early_stopping_config = eval_config.get('early_stopping', {})
        self.early_stopping_enabled = early_stopping_config.get('enabled', False)
        self.early_stopping_patience = early_stopping_config.get('patience', 3)
        self.early_stopping_metric = early_stopping_config.get('metric', 'eval_loss')
        self.early_stopping_mode = early_stopping_config.get('mode', 'min')
        
        # Logging settings
        logging_config = config.get('logging', {})
        self.logging_steps = logging_config.get('log_steps', 100)
        self.use_wandb = logging_config.get('use_wandb', False)
        self.wandb_project = logging_config.get('wandb_project', 'transformer_model_training')
        self.wandb_run_name = None
        
        # Set device
        self.device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
        
        # Initialize step counters
        self.global_step = 0
        self.epoch = 0
        
        # Initialize early stopping variables
        self.best_metric = float('inf') if self.early_stopping_mode == 'min' else float('-inf')
        self.no_improvement_count = 0


class Trainer:
    """
    Trainer class for model training.
    """
    
    def __init__(
        self,
        model: nn.Module,
        tokenizer: Any,
        train_dataset: Optional[Dataset] = None,
        eval_dataset: Optional[Dataset] = None,
        args: Optional[TrainingArguments] = None,
        data_collator: Optional[Callable] = None,
        compute_metrics_fn: Optional[Callable] = None,
    ):
        """
        Initialize trainer.
        
        Args:
            model: Model to train
            tokenizer: Tokenizer for the model
            train_dataset: Training dataset
            eval_dataset: Evaluation dataset
            args: Training arguments
            data_collator: Function to collate data samples into batches
            compute_metrics_fn: Function to compute evaluation metrics
        """
        self.model = model
        self.tokenizer = tokenizer
        self.train_dataset = train_dataset
        self.eval_dataset = eval_dataset
        self.args = args or TrainingArguments({}, "default")
        self.data_collator = data_collator or self._default_collator
        self.compute_metrics_fn = compute_metrics_fn
        
        # Set up optimizer
        self.optimizer = self._create_optimizer()
        
        # Set up learning rate scheduler
        self.lr_scheduler = self._create_scheduler()
        
        # Set up mixed precision training
        self.scaler = torch.cuda.amp.GradScaler() if self.args.mixed_precision == 'fp16' else None
        
        # Move model to device
        self.model.to(self.args.device)
        
        # Enable gradient checkpointing if requested
        if self.args.gradient_checkpointing and hasattr(self.model, 'gradient_checkpointing_enable'):
            self.model.gradient_checkpointing_enable()
    
    def _create_optimizer(self) -> torch.optim.Optimizer:
        """
        Create optimizer.
        
        Returns:
            Optimizer instance
        """
        # Get optimizer parameters
        no_decay = ['bias', 'LayerNorm.weight']
        optimizer_grouped_parameters = [
            {
                'params': [p for n, p in self.model.named_parameters() if not any(nd in n for nd in no_decay)],
                'weight_decay': self.args.weight_decay,
            },
            {
                'params': [p for n, p in self.model.named_parameters() if any(nd in n for nd in no_decay)],
                'weight_decay': 0.0,
            },
        ]
        
        # Create optimizer based on name
        if self.args.optimizer_name == 'adamw':
            return torch.optim.AdamW(
                optimizer_grouped_parameters,
                lr=self.args.learning_rate,
                betas=(self.args.beta1, self.args.beta2),
                eps=self.args.eps
            )
        elif self.args.optimizer_name == 'adam':
            return torch.optim.Adam(
                optimizer_grouped_parameters,
                lr=self.args.learning_rate,
                betas=(self.args.beta1, self.args.beta2),
                eps=self.args.eps
            )
        elif self.args.optimizer_name == 'sgd':
            return torch.optim.SGD(
                optimizer_grouped_parameters,
                lr=self.args.learning_rate,
                momentum=0.9
            )
        else:
            logger.warning(f"Unsupported optimizer: {self.args.optimizer_name}. Using AdamW.")
            return torch.optim.AdamW(
                optimizer_grouped_parameters,
                lr=self.args.learning_rate,
                betas=(self.args.beta1, self.args.beta2),
                eps=self.args.eps
            )
    
    def _create_scheduler(self) -> Optional[torch.optim.lr_scheduler._LRScheduler]:
        """
        Create learning rate scheduler.
        
        Returns:
            Learning rate scheduler instance
        """
        # Estimate number of training steps
        if self.train_dataset is None:
            return None
        
        num_update_steps_per_epoch = max(1, len(self.train_dataset) // (self.args.train_batch_size * self.args.gradient_accumulation_steps))
        num_training_steps = self.args.epochs * num_update_steps_per_epoch
        
        # Create scheduler based on schedule name
        if self.args.lr_schedule == 'linear':
            return torch.optim.lr_scheduler.LinearLR(
                self.optimizer,
                start_factor=1.0,
                end_factor=self.args.min_learning_rate / self.args.learning_rate,
                total_iters=num_training_steps - self.args.warmup_steps
            )
        elif self.args.lr_schedule == 'cosine':
            return torch.optim.lr_scheduler.CosineAnnealingLR(
                self.optimizer,
                T_max=num_training_steps - self.args.warmup_steps,
                eta_min=self.args.min_learning_rate
            )
        elif self.args.lr_schedule == 'constant':
            return torch.optim.lr_scheduler.ConstantLR(
                self.optimizer,
                factor=1.0,
                total_iters=num_training_steps
            )
        else:
            logger.warning(f"Unsupported learning rate schedule: {self.args.lr_schedule}. Using linear.")
            return torch.optim.lr_scheduler.LinearLR(
                self.optimizer,
                start_factor=1.0,
                end_factor=self.args.min_learning_rate / self.args.learning_rate,
                total_iters=num_training_steps - self.args.warmup_steps
            )
    
    def _default_collator(self, examples: List[Dict]) -> Dict[str, torch.Tensor]:
        """
        Default data collator.
        
        Args:
            examples: List of examples
            
        Returns:
            Batch dictionary
        """
        # Determine text column
        text_column = None
        if 'text' in examples[0]:
            text_column = 'text'
        elif 'content' in examples[0]:
            text_column = 'content'
        elif 'sentence' in examples[0]:
            text_column = 'sentence'
        
        if text_column is None:
            logger.warning("No text column found in examples")
            return {}
        
        # Tokenize texts
        texts = [example[text_column] for example in examples]
        batch = self.tokenizer(
            texts,
            padding=True,
            truncation=True,
            return_tensors="pt"
        )
        
        # Add labels for language modeling
        batch["labels"] = batch["input_ids"].clone()
        
        return batch
    
    def _get_train_dataloader(self) -> DataLoader:
        """
        Get training dataloader.
        
        Returns:
            Training dataloader
        """
        if self.train_dataset is None:
            raise ValueError("Trainer: training requires a train_dataset")
        
        return DataLoader(
            self.train_dataset,
            batch_size=self.args.train_batch_size,
            shuffle=True,
            collate_fn=self.data_collator,
            num_workers=0,
            pin_memory=True,
            drop_last=True
        )
    
    def _get_eval_dataloader(self) -> Optional[DataLoader]:
        """
        Get evaluation dataloader.
        
        Returns:
            Evaluation dataloader
        """
        if self.eval_dataset is None:
            return None
        
        return DataLoader(
            self.eval_dataset,
            batch_size=self.args.eval_batch_size,
            shuffle=False,
            collate_fn=self.data_collator,
            num_workers=0,
            pin_memory=True
        )
    
    def _prepare_inputs(self, inputs: Dict[str, torch.Tensor]) -> Dict[str, torch.Tensor]:
        """
        Prepare inputs for the model.
        
        Args:
            inputs: Input dictionary
            
        Returns:
            Prepared inputs
        """
        # Move tensors to device
        for k, v in inputs.items():
            if isinstance(v, torch.Tensor):
                inputs[k] = v.to(self.args.device)
        
        return inputs
    
    def train(self) -> Dict[str, float]:
        """
        Train the model.
        
        Returns:
            Dictionary with training metrics
        """
        # Set model to training mode
        self.model.train()
        
        # Get dataloader
        train_dataloader = self._get_train_dataloader()
        
        # Initialize training metrics
        total_loss = 0.0
        epoch_loss = 0.0
        step_loss = 0.0
        step_count = 0
        start_time = time.time()
        
        # Training loop
        logger.info(f"Starting training for {self.args.epochs} epochs")
        
        for epoch in range(self.args.epochs):
            self.args.epoch = epoch
            epoch_start_time = time.time()
            epoch_loss = 0.0
            step_count = 0
            
            # Iterate over batches
            progress_bar = tqdm(train_dataloader, desc=f"Epoch {epoch+1}/{self.args.epochs}")
            
            for step, inputs in enumerate(progress_bar):
                # Prepare inputs
                inputs = self._prepare_inputs(inputs)
                
                # Forward pass with mixed precision if enabled
                if self.args.mixed_precision == 'fp16':
                    with torch.cuda.amp.autocast():
                        outputs = self.model(**inputs)
                        loss = outputs.loss / self.args.gradient_accumulation_steps
                    
                    # Backward pass with gradient scaling
                    self.scaler.scale(loss).backward()
                else:
                    outputs = self.model(**inputs)
                    loss = outputs.loss / self.args.gradient_accumulation_steps
                    loss.backward()
                
                # Update weights if gradient accumulation steps reached
                if (step + 1) % self.args.gradient_accumulation_steps == 0 or step == len(train_dataloader) - 1:
                    # Clip gradients
                    if self.args.gradient_clipping > 0:
                        if self.args.mixed_precision == 'fp16':
                            self.scaler.unscale_(self.optimizer)
                        
                        torch.nn.utils.clip_grad_norm_(self.model.parameters(), self.args.gradient_clipping)
                    
                    # Update weights
                    if self.args.mixed_precision == 'fp16':
                        self.scaler.step(self.optimizer)
                        self.scaler.update()
                    else:
                        self.optimizer.step()
                    
                    # Update learning rate
                    if self.lr_scheduler is not None:
                        self.lr_scheduler.step()
                    
                    # Zero gradients
                    self.optimizer.zero_grad()
                    
                    # Update global step
                    self.args.global_step += 1
                
                # Update metrics
                step_loss = loss.item() * self.args.gradient_accumulation_steps
                epoch_loss += step_loss
                total_loss += step_loss
                step_count += 1
                
                # Update progress bar
                progress_bar.set_postfix({
                    "loss": step_loss,
                    "avg_loss": epoch_loss / (step + 1),
                    "lr": self.optimizer.param_groups[0]['lr']
                })
                
                # Evaluate if needed
                if self.args.eval_steps > 0 and self.args.global_step > 0 and self.args.global_step % self.args.eval_steps == 0:
                    eval_results = self.evaluate()
                    self.model.train()
                    
                    # Log evaluation results
                    logger.info(f"Evaluation results at step {self.args.global_step}: {eval_results}")
                    
                    # Check for early stopping
                    if self.args.early_stopping_enabled and self._check_early_stopping(eval_results):
                        logger.info(f"Early stopping triggered at step {self.args.global_step}")
                        return {
                            "loss": total_loss / step_count,
                            "epoch": epoch,
                            "learning_rate": self.optimizer.param_groups[0]['lr'],
                            "train_time": time.time() - start_time
                        }
                
                # Save checkpoint if needed
                if self.args.save_steps > 0 and self.args.global_step > 0 and self.args.global_step % self.args.save_steps == 0:
                    self.save_checkpoint()
            
            # End of epoch
            epoch_time = time.time() - epoch_start_time
            logger.info(f"Epoch {epoch+1} completed in {epoch_time:.2f}s, loss: {epoch_loss/step_count:.4f}")
            
            # Evaluate at the end of each epoch
            eval_results = self.evaluate()
            self.model.train()
            
            # Log evaluation results
            logger.info(f"Evaluation results at epoch {epoch+1}: {eval_results}")
            
            # Save checkpoint at the end of each epoch
            self.save_checkpoint()
            
            # Check for early stopping
            if self.args.early_stopping_enabled and self._check_early_stopping(eval_results):
                logger.info(f"Early stopping triggered at epoch {epoch+1}")
                break
        
        # End of training
        train_time = time.time() - start_time
        logger.info(f"Training completed in {train_time:.2f}s, avg loss: {total_loss/step_count:.4f}")
        
        # Save final model
        self.save_checkpoint(is_final=True)
        
        return {
            "loss": total_loss / step_count,
            "epoch": self.args.epoch,
            "learning_rate": self.optimizer.param_groups[0]['lr'],
            "train_time": train_time
        }
    
    def evaluate(self) -> Dict[str, float]:
        """
        Evaluate the model.
        
        Returns:
            Dictionary with evaluation metrics
        """
        # Set model to evaluation mode
        self.model.eval()
        
        # Get dataloader
        eval_dataloader = self._get_eval_dataloader()
        if eval_dataloader is None:
            return {}
        
        # Initialize evaluation metrics
        total_loss = 0.0
        step_count = 0
        all_preds = []
        all_labels = []
        
        # Evaluation loop
        with torch.no_grad():
            for step, inputs in enumerate(eval_dataloader):
                # Prepare inputs
                inputs = self._prepare_inputs(inputs)
                
                # Forward pass
                outputs = self.model(**inputs)
                loss = outputs.loss
                logits = outputs.logits
                
                # Update metrics
                total_loss += loss.item()
                step_count += 1
                
                # Store predictions and labels for metrics
                if "labels" in inputs:
                    preds = torch.argmax(logits, dim=-1)
                    all_preds.append(preds.detach().cpu())
                    all_labels.append(inputs["labels"].detach().cpu())
        
        # Compute average loss
        avg_loss = total_loss / step_count if step_count > 0 else 0
        
        # Compute perplexity
        perplexity = math.exp(avg_loss)
        
        # Compute additional metrics if available
        metrics = {
            "eval_loss": avg_loss,
            "perplexity": perplexity
        }
        
        # Compute custom metrics if function is provided
        if self.compute_metrics_fn is not None and all_preds and all_labels:
            # Concatenate predictions and labels
            all_preds = torch.cat(all_preds, dim=0)
            all_labels = torch.cat(all_labels, dim=0)
            
            # Compute metrics
            additional_metrics = self.compute_metrics_fn(all_preds, all_labels)
            metrics.update(additional_metrics)
        
        return metrics
    
    def _check_early_stopping(self, eval_results: Dict[str, float]) -> bool:
        """
        Check if early stopping criteria are met.
        
        Args:
            eval_results: Evaluation results
            
        Returns:
            Whether to stop training
        """
        if not self.args.early_stopping_enabled:
            return False
        
        # Get metric value
        metric = eval_results.get(self.args.early_stopping_metric)
        if metric is None:
            return False
        
        # Check if metric improved
        improved = False
        if self.args.early_stopping_mode == 'min':
            improved = metric < self.args.best_metric
        else:
            improved = metric > self.args.best_metric
        
        if improved:
            self.args.best_metric = metric
            self.args.no_improvement_count = 0
            return False
        else:
            self.args.no_improvement_count += 1
            return self.args.no_improvement_count >= self.args.early_stopping_patience
    
    def save_checkpoint(self, is_final: bool = False) -> None:
        """
        Save model checkpoint.
        
        Args:
            is_final: Whether this is the final checkpoint
        """
        # Create checkpoint directory
        checkpoint_dir = os.path.join(
            self.args.output_dir,
            f"checkpoint-{self.args.global_step}" if not is_final else "final-model"
        )
        os.makedirs(checkpoint_dir, exist_ok=True)
        
        # Save model and tokenizer
        self.model.save_pretrained(checkpoint_dir)
        self.tokenizer.save_pretrained(checkpoint_dir)
        
        # Save optimizer state if requested
        if self.args.save_optimizer_state:
            optimizer_path = os.path.join(checkpoint_dir, "optimizer.pt")
            torch.save(self.optimizer.state_dict(), optimizer_path)
        
        # Save training arguments
        args_path = os.path.join(checkpoint_dir, "training_args.json")
        with open(args_path, 'w') as f:
            import json
            json.dump(vars(self.args), f, indent=2, default=str)
        
        logger.info(f"Saved checkpoint to {checkpoint_dir}")
        
        # Remove old checkpoints if needed
        if self.args.keep_last_n > 0 and not is_final:
            self._cleanup_checkpoints()
    
    def _cleanup_checkpoints(self) -> None:
        """
        Remove old checkpoints to save disk space.
        """
        # Get all checkpoint directories
        checkpoint_dirs = [d for d in os.listdir(self.args.output_dir) if d.startswith("checkpoint-")]
        
        # Sort by step number
        checkpoint_dirs.sort(key=lambda x: int(x.split("-")[1]))
        
        # Remove old checkpoints
        if len(checkpoint_dirs) > self.args.keep_last_n:
            for old_dir in checkpoint_dirs[:-self.args.keep_last_n]:
                old_path = os.path.join(self.args.output_dir, old_dir)
                import shutil
                shutil.rmtree(old_path)
                logger.info(f"Removed old checkpoint: {old_path}")

In [ ]:
%%writefile src/utils/tokenization.py
"""
Tokenization utilities for the AI model training workflow.
"""

import logging
from typing import Dict, Any, Optional

# Try to import optional dependencies
try:
    from transformers import AutoTokenizer
    TRANSFORMERS_AVAILABLE = True
except ImportError:
    TRANSFORMERS_AVAILABLE = False

# Configure logging
logger = logging.getLogger(__name__)


def get_tokenizer(config: Dict) -> Any:
    """
    Get tokenizer based on configuration.
    
    Args:
        config: Tokenizer configuration
        
    Returns:
        Tokenizer instance
    """
    if not TRANSFORMERS_AVAILABLE:
        logger.error("Transformers library is not available. Please install with: pip install transformers")
        return None
    
    tokenizer_type = config.get('type', 'huggingface')
    
    if tokenizer_type == 'huggingface':
        # Get tokenizer name
        tokenizer_name = config.get('name', 'gpt2')
        
        # Get additional parameters
        padding_side = config.get('padding_side', 'right')
        truncation_side = config.get('truncation_side', 'right')
        max_length = config.get('max_length', 1024)
        add_bos_token = config.get('add_bos_token', False)
        add_eos_token = config.get('add_eos_token', False)
        
        # Create tokenizer
        tokenizer = AutoTokenizer.from_pretrained(tokenizer_name)
        
        # Configure tokenizer
        tokenizer.padding_side = padding_side
        tokenizer.truncation_side = truncation_side
        tokenizer.model_max_length = max_length
        
        # Add special tokens if needed
        special_tokens = {}
        if add_bos_token and tokenizer.bos_token is None:
            special_tokens['bos_token'] = '<bos>'
        if add_eos_token and tokenizer.eos_token is None:
            special_tokens['eos_token'] = '<eos>'
        
        if special_tokens:
            tokenizer.add_special_tokens(special_tokens)
        
        logger.info(f"Created tokenizer: {tokenizer_name}")
        return tokenizer
    else:
        logger.error(f"Unsupported tokenizer type: {tokenizer_type}")
        return None

In [ ]:
%%writefile src/utils/metrics.py
"""
Evaluation metrics for the AI model training workflow.
"""

import torch
import numpy as np
from typing import Dict, Any, Tuple


def compute_metrics(predictions: torch.Tensor, labels: torch.Tensor) -> Dict[str, float]:
    """
    Compute evaluation metrics.
    
    Args:
        predictions: Model predictions
        labels: Ground truth labels
        
    Returns:
        Dictionary with metric results
    """
    # Convert to numpy arrays
    if isinstance(predictions, torch.Tensor):
        predictions = predictions.cpu().numpy()
    if isinstance(labels, torch.Tensor):
        labels = labels.cpu().numpy()
    
    # Compute accuracy
    accuracy = np.mean(predictions == labels)
    
    return {
        "accuracy": float(accuracy)
    }

## Create a Simple Training Script

Now, let's create a simplified training script for our demo.

In [ ]:
%%writefile colab_train.py
#!/usr/bin/env python
"""
Simplified training script for Colab/Kaggle demo.
"""

import os
import logging
import yaml
import torch
from typing import Dict

from src.data.loaders import DatasetLoader
from src.data.preprocessors import DataPreprocessor
from src.model.architecture import create_model_from_config
from src.model.training import Trainer, TrainingArguments
from src.utils.tokenization import get_tokenizer

# Configure logging
logging.basicConfig(
    level=logging.INFO,
    format='%(asctime)s - %(name)s - %(levelname)s - %(message)s'
)
logger = logging.getLogger(__name__)


def load_config(config_path: str) -> Dict:
    """Load configuration from YAML file."""
    try:
        with open(config_path, 'r') as f:
            config = yaml.safe_load(f)
        return config
    except Exception as e:
        logger.error(f"Error loading configuration: {e}")
        raise


def main():
    """Main function for training."""
    # Load configuration
    config_path = "config/simple_config.yaml"
    config = load_config(config_path)
    
    # Set random seed
    torch.manual_seed(config['seed'])
    
    # Create output directory
    os.makedirs(config['output_dir'], exist_ok=True)
    
    # Get active stage
    active_stage = config['training']['active_stage']
    logger.info(f"Active training stage: {active_stage}")
    
    # Get stage configuration
    stage_config = None
    for stage in config['training']['stages']:
        if stage['name'] == active_stage:
            stage_config = stage
            break
    
    if stage_config is None:
        raise ValueError(f"Training stage {active_stage} not found in configuration")
    
    # Initialize dataset loader
    dataset_loader = DatasetLoader(config_path)
    
    # Load datasets
    train_dataset = None
    eval_dataset = None
    
    for dataset_config in stage_config['datasets']:
        if dataset_config.get('split') == 'train':
            train_dataset = dataset_loader.load_dataset(
                dataset_config['name'],
                subset=dataset_config.get('subset'),
                split='train',
                streaming=dataset_config.get('streaming', False),
                max_samples=dataset_config.get('max_samples')
            )
        elif dataset_config.get('split') == 'test' or dataset_config.get('split') == 'validation':
            eval_dataset = dataset_loader.load_dataset(
                dataset_config['name'],
                subset=dataset_config.get('subset'),
                split=dataset_config.get('split'),
                streaming=dataset_config.get('streaming', False),
                max_samples=dataset_config.get('max_samples')
            )
    
    # Initialize data preprocessor
    preprocessor = DataPreprocessor(config)
    
    # Preprocess datasets
    if train_dataset:
        train_dataset = preprocessor.process_dataset(train_dataset)
    if eval_dataset:
        eval_dataset = preprocessor.process_dataset(eval_dataset)
    
    # Get tokenizer
    tokenizer = get_tokenizer(config['tokenizer'])
    
    # Create model
    model = create_model_from_config(config)
    
    # Create training arguments
    training_args = TrainingArguments(config, active_stage)
    
    # Create trainer
    trainer = Trainer(
        model=model,
        tokenizer=tokenizer,
        train_dataset=train_dataset,
        eval_dataset=eval_dataset,
        args=training_args
    )
    
    # Train model
    results = trainer.train()
    
    logger.info(f"Training results: {results}")
    
    # Evaluate final model
    eval_results = trainer.evaluate()
    logger.info(f"Final evaluation results: {eval_results}")
    
    logger.info("Training completed successfully")


if __name__ == "__main__":
    main()

## Run the Training Script

Now, let's run the training script to train a small model on the IMDB dataset.

In [ ]:
# Make the script executable
!chmod +x colab_train.py

In [ ]:
# Run the training script
!python colab_train.py

## Generate Text with the Trained Model

Let's use our trained model to generate some text.

In [ ]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer

# Load the trained model and tokenizer
model_path = "output/final-model"
model = AutoModelForCausalLM.from_pretrained(model_path)
tokenizer = AutoTokenizer.from_pretrained(model_path)

# Set the model to evaluation mode
model.eval()

# Function to generate text
def generate_text(prompt, max_length=100, num_return_sequences=1):
    # Tokenize the prompt
    input_ids = tokenizer(prompt, return_tensors="pt").input_ids
    
    # Generate text
    with torch.no_grad():
        outputs = model.generate(
            input_ids,
            max_length=max_length,
            num_return_sequences=num_return_sequences,
            temperature=0.7,
            top_p=0.9,
            do_sample=True,
            pad_token_id=tokenizer.eos_token_id
        )
    
    # Decode the generated text
    generated_texts = [tokenizer.decode(output, skip_special_tokens=True) for output in outputs]
    
    return generated_texts

# Generate text with a prompt
prompt = "This movie was really"
generated_texts = generate_text(prompt, max_length=50, num_return_sequences=3)

# Print the generated texts
for i, text in enumerate(generated_texts):
    print(f"Generated text {i+1}:\n{text}\n")

## Using the Advanced Features

The full training pipeline includes many advanced features that you can use by importing the appropriate modules. Here's a quick example of how to use the hyperparameter tuning module:

In [ ]:
# Install Ray Tune if you haven't already
!pip install "ray[tune]"

In [ ]:
# Import the hyperparameter tuning module
from src.utils.hyperparameter_tuning import optimize_hyperparameters

# Define a search space
search_space = {
    "learning_rate": tune.loguniform(1e-5, 1e-3),
    "weight_decay": tune.loguniform(1e-6, 1e-3),
    "batch_size": tune.choice([8, 16, 32]),
    "dropout": tune.uniform(0.1, 0.5),
}

# Run hyperparameter optimization (this will take some time)
# best_params = optimize_hyperparameters(
#     config_path="config/simple_config.yaml",
#     search_space=search_space,
#     num_samples=5,  # Reduced for demo
#     num_epochs=1,   # Reduced for demo
#     search_alg="hyperopt",
#     scheduler="asha",
#     metric="eval_loss",
#     mode="min"
# )

# print(f"Best hyperparameters: {best_params['best_config']}")

## Conclusion

This notebook demonstrates how to use the advanced machine learning training pipeline in Google Colab or Kaggle. The pipeline includes components for data loading, preprocessing, augmentation, model architecture, training, evaluation, hyperparameter tuning, and model optimization.

For more advanced features, refer to the full documentation in the repository's README.md file.